## Pairwise grading

In order to assess the specificity of certain disclosures, we are going to perform pairwise grading. This means starting with the positively identified samples and then putting them up against one another in a grading schema (based on categories from Oppong-Tawiah and Webster 2023)

In [ ]:
import sys
sys.path.append('..')
import utils
from path import Path
import openai
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get OpenRouter API key from environment variables
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
if not OPENROUTER_API_KEY:
    raise ValueError("Please set OPENROUTER_API_KEY in your .env file or environment variables")


In [5]:
results_path = '../results/SAO_RAG_RESULTS'
results_path = Path(results_path)

In [6]:
results = utils.load_company_data(results_path)

In [7]:
results

,company,original_index,year,text,climate_litigation,retrieval_similarity,num_examples_used
0,RCL,25,2014,We believe that the impact of ships on the glo...,climate_litigation: 0,0.564889,5
1,RCL,48,2014,To the extent the tonnage tax laws of these co...,climate_litigation: 0,0.531344,5
2,RCL,40,2014,"dollar, including, among others, the British p...",climate_litigation: 0,0.528860,5
3,RCL,24,2014,"The ISM Code is mandatory for all vessels, inc...",climate_litigation: 0,0.523801,5
4,RCL,41,2014,An increase in fuel prices not only impacts ou...,climate_litigation: 0,0.516437,5
...,...,...,...,...,...,...,...
52505,GLP,64,2016,"For example, our partnership agreement: ·\nper...",climate_litigation: 0,0.484702,5
52506,GLP,88,2016,We operate our business under three segments: ...,climate_litigation: 0,0.484497,5
52507,GLP,104,2016,Our gross profit for 2014 was negatively impac...,climate_litigation: 0,0.482937,5
52508,GLP,285,2016,"The fees, which are based upon an estimate of ...",climate_litigation: 0,0.482671,5


Extracting the binary number from the climate litigation assignment 

In [8]:
trouble_indices = []

In [9]:
results['climate_litigation_binary'] = results.apply(
    lambda row: utils.extract_flag(row['climate_litigation'], row.name, trouble_indices), 
    axis=1
)

Subsetting to only the positively identified phrases

In [10]:
results_1 = results[results['climate_litigation_binary'] == 1]

In [11]:
len(results_1)

616

In [12]:
#calculate the number of pairwise comparisons we need to make; note this is mn(n−1) rather than mn(n−1)/2 because we always need to swap the presentation order of the responses si and sj
#reference: Gao 2025 

total_comparisons = len(results_1) * (len(results_1) - 1)
total_comparisons


378840

## Pairwise prompting

In [13]:
client = openai.OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

In [14]:
def get_pairwise_prompt(sentence_1, sentence_2):
    return (
        "You are comparing two sections to decide which is MORE SPECIFIC about CLIMATE CHANGE LITIGATION.\n"
        "\n"
        "Use the literature notion of specificity: level of detail and precision about place, time, numbers, or descriptive/sensory facts "
        "('spatio-temporal/descriptive' vs. generalized wording). Apply this ONLY to climate change litigation content.\n"
        "\n"
        "PROCESS\n"
        "1) Independently assess Section 1, then Section 2.\n"
        "2) For each section, judge specificity based on:\n"
        "   - PRECISION of anchors: named parties/case title/docket, forum or jurisdiction, statute/regulation/policy, procedural posture, dates/time frames, monetary/quantified remedies or thresholds.\n"
        "   - LINKAGE: whether those anchors are clearly tied to climate claims (not just generic legal or sustainability talk).\n"
        "   - FALSIFIABILITY: whether the details are concrete enough to be checked or challenged (e.g., exact court, filing date, cited rule).\n"
        "   - CONTEXTUAL CLARITY: whether the details fit together coherently (who did what, where, when, under which rule, seeking what outcome).\n"
        "Ignore specificity about non-climate topics.\n"
        "\n"
        "SCORING GUIDELINES (no counting—judge by presence, precision, and linkage strength):\n"
        "0: No climate litigation content at all.\n"
        "10: Climate issues mentioned but not litigation.\n"
        "20: Litigation is referenced but only in generic terms; no precise legal anchor or linkage to climate claims.\n"
        "40: At least one legal anchor appears (e.g., forum, party, statute) but is vague or only loosely tied to climate claims; timing/place/amounts are broad or implied.\n"
        "60: Clear, specific legal anchor(s) precisely linked to climate claims (e.g., named agency action, identifiable forum or statute, plausible timeframe or remedy), but some elements remain general or implicit.\n"
        "80: Multiple precise and well-linked anchors (e.g., named party or case title AND specific forum/jurisdiction AND concrete timeframe or remedy); coherent and checkable, minor gaps only.\n"
        "90–100: Highly specific and verifiable (e.g., case name or docket; exact court; specific date(s) or stage; cited rule/statute; defined remedy/amounts), tightly and explicitly tied to climate claims with strong internal coherence.\n"
        "Choose an integer score that best fits the description above.\n"
        "\n"
        "DECISION\n"
        "- After scoring both independently, choose the MORE SPECIFIC section.\n"
        "- If scores differ by ≥20, pick the higher score.\n"
        "- If |diff| ≤ 10, apply tie-breakers IN ORDER:\n"
        "  (i) Prefer the section that names a case, docket, or parties with a specific forum/jurisdiction.\n"
        "  (ii) Prefer the section with clearer procedural posture and explicit linkage to a climate rule/policy/statute.\n"
        "  (iii) Prefer the section with more precise spatio-temporal or monetary details that are checkable.\n"
        "  (iv) If still tied, output \"section 2\" (bias control).\n"
        "\n"
        "RULES\n"
        "- OUTPUT MUST BE VALID JSON ONLY (no extra text, no comments, no trailing commas).\n"
        "\n"
        "OUTPUT FORMAT (exact schema, do not provide any other text):\n"
        "[\n"
        "  {\n"
        "    \"binary result\": \"section 1\" or \"section 2\",\n"
        "    \"section_1\": <integer 0-100>,\n"
        "    \"section_2\": <integer 0-100>\n"
        "  }\n"
        "]\n"
        "\n"
        "SECTIONS\n"
        f"Section 1: {sentence_1}\n"
        f"Section 2: {sentence_2}\n"
    )

In [15]:
sentence_1 = results_1.iloc[13]['text']
sentence_2 = results_1.iloc[16]['text']
message = get_pairwise_prompt(sentence_1, sentence_2)
from pprint import pprint
pprint(message)

('You are comparing two sections to decide which is MORE SPECIFIC about '
 'CLIMATE CHANGE LITIGATION.\n'
 '\n'
 'Use the literature notion of specificity: level of detail and precision '
 'about place, time, numbers, or descriptive/sensory facts '
 "('spatio-temporal/descriptive' vs. generalized wording). Apply this ONLY to "
 'climate change litigation content.\n'
 '\n'
 'PROCESS\n'
 '1) Independently assess Section 1, then Section 2.\n'
 '2) For each section, judge specificity based on:\n'
 '   - PRECISION of anchors: named parties/case title/docket, forum or '
 'jurisdiction, statute/regulation/policy, procedural posture, dates/time '
 'frames, monetary/quantified remedies or thresholds.\n'
 '   - LINKAGE: whether those anchors are clearly tied to climate claims (not '
 'just generic legal or sustainability talk).\n'
 '   - FALSIFIABILITY: whether the details are concrete enough to be checked '
 'or challenged (e.g., exact court, filing date, cited rule).\n'
 '   - CONTEXTUAL CLARI

In [16]:
SYSTEM_MESSAGE = (
    "You are a legal and environmental disclosure expert. Previously, someone else extracted all of the sentences from several companies' annual reports that related to climate litigation. Now, we are interested in assessing the specificity of these disclosures.\n\n"
    "Specificity captures the level of detail and precision, measured by the number of specific details related to place, time, numbers, or the five senses (i.e., “descriptive” and “spatio-temporal” words), as opposed to “generalized words”.\n\n"
    "The most important part of your job is choosing which section is more specific. So focus on that first and foremost. You only respond in the JSON format, no other text. EVER."
)

In [17]:
response = client.chat.completions.create(
    model="meta-llama/llama-3.1-70b-instruct",
    messages=[
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": message}
    ],
    temperature=0.0
)
print(response.choices[0].message.content)


[
  {
    "binary result": "section 1",
    "section_1": 60,
    "section_2": 40
  }
]


I'm having the problem that if I assess with the more powerful LLM it is more consistent but the one that proved most accurate in our other tasks is failing, always saying section 2 (mostly) and also struggling in consistency. Maybe it needs the guardrails of few shot learning to be effective? I wonder if I could justify a different model in the pairwise grading than in the original work. 

We can justify from Gao et al. 2025 that the best open source model for pairwise grading is meta-llama/llama-3.1-70b-instruct. The problem is just then showing why we did what we did in the first task. I think that's doable, we just need to cast the original problem as it was, a more straightforward classification task that should be achievable for all LLMs. 

## Running a small test against state of the art GPTo5

Think of this as our 'groundtruth' since it has been shown to be so successful on these types of tasks